# Field Report Image Processing (Gemini Vision Edition)

This notebook automates the extraction, classification, and reporting of field operation photos.
It uses Google's state-of-the-art **Gemini 2.5 Flash** vision model to classify images with >95% accuracy.

### Instructions:
1. Upload your target photos (or ZIPs) and `Prefix.txt` to a folder in your Google Drive.
2. Add your Gemini API Key in the Colab Secrets tab (key icon on the left) with the name `GEMINI_API_KEY`.
3. Run all cells (`Runtime -> Run all`).

In [ ]:
!pip install -q -U google-generativeai
!pip install -q openpyxl pandas pillow

In [ ]:
import os
import re
import io
import gc
import zipfile
import time
from datetime import datetime
import pandas as pd
from PIL import Image as PILImage
from PIL.ExifTags import TAGS
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from google.colab import drive
from google.colab import userdata
from google.colab import files
import google.generativeai as genai

# ==========================================
# 1. MOUNT GOOGLE DRIVE
# ==========================================
drive.mount('/content/drive')

# ==========================================
# 2. CONFIGURATION & PATHS
# ==========================================
# Update this path to where your photos/zips are stored in Google Drive:
TARGET_WORKSPACE = '/content/drive/MyDrive/ImageProcessing'

RESULT_DIR = os.path.join(TARGET_WORKSPACE, 'Result')
PREFIX_FILE = os.path.join(TARGET_WORKSPACE, 'Prefix.txt')

os.makedirs(RESULT_DIR, exist_ok=True)
EXTRACT_PHOTOS_DIR = os.path.join(RESULT_DIR, 'Extracted_Photos')
THUMB_CACHE_DIR = os.path.join(RESULT_DIR, 'temp_thumbnails')
os.makedirs(EXTRACT_PHOTOS_DIR, exist_ok=True)
os.makedirs(THUMB_CACHE_DIR, exist_ok=True)


# ==========================================
# 2.5 ENABLE SYSTEM LOGGING
# ==========================================
import sys

log_file_path = os.path.join(RESULT_DIR, f"debug_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
class Logger(object):
    def __init__(self, original_stream, log_file):
        self.terminal = original_stream
        self.log = open(log_file, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger(sys.stdout, log_file_path)
sys.stderr = Logger(sys.stderr, log_file_path)
print(f"📝 Logging all outputs to: {log_file_path}")

print(f"📂 Target Workspace: {TARGET_WORKSPACE}")
print(f"💾 Results will be saved to: {RESULT_DIR}")

# ==========================================
# 3. CONFIGURE GEMINI VISION MODEL
# ==========================================
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    # Gemini 2.5 Flash provides extreme speed and >95% accuracy for this task
    model = genai.GenerativeModel('gemini-2.5-flash')
    ML_AVAILABLE = True
    print("✅ Successfully authenticated with Gemini API!")
except Exception as e:
    print("⚠️ WARNING: Could not configure Gemini. Ensure GEMINI_API_KEY is set in Colab Secrets.")
    print(f"Details: {e}")
    ML_AVAILABLE = False


In [ ]:
# ==========================================
# 4. HELPER FUNCTIONS
# ==========================================
def extract_camera_capture_date(raw_bytes, local_filepath=None):
    try:
        img = PILImage.open(io.BytesIO(raw_bytes))
        exif_data = img._getexif()
        if exif_data:
            for tag_id, value in exif_data.items():
                tag_name = TAGS.get(tag_id, tag_id)
                if tag_name == 'DateTimeOriginal' and value:
                    dt = datetime.strptime(value.strip(), "%Y:%m:%d %H:%M:%S")
                    return dt.strftime("%d-%m-%Y %H:%M")
    except Exception: pass
    if local_filepath and os.path.exists(local_filepath):
        try:
            timestamp = os.path.getmtime(local_filepath)
            return datetime.fromtimestamp(timestamp).strftime("%d-%m-%Y %H:%M")
        except Exception: pass
    return "Unknown Date"

def create_compressed_thumbnail(raw_bytes, target_path):
    try:
        img = PILImage.open(io.BytesIO(raw_bytes))
        if img.mode in ('RGBA', 'LA'):
            background = PILImage.new('RGB', img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[3])
            img = background
        elif img.mode != 'RGB':
            img = img.convert('RGB')
        img.thumbnail((120, 90), PILImage.Resampling.LANCZOS)
        img.save(target_path, 'JPEG', quality=50, optimize=True)
        return True
    except Exception: return False

def save_optimized_photo(raw_bytes, export_filepath):
    try:
        img = PILImage.open(io.BytesIO(raw_bytes))
        if img.mode != 'RGB': img = img.convert('RGB')
        img.save(export_filepath, 'JPEG', quality=70, optimize=True)
    except Exception:
        with open(export_filepath, 'wb') as out_f: out_f.write(raw_bytes)

def get_container_overrides(path_string):
    label_text = str(path_string).upper()
    override = {'band': None, 'hardware': None}
    if "2600" in label_text: override['band'] = "2600"
    if "AAU" in label_text: override['hardware'] = "AAU"
    return override

def extract_site_id(trace_path, site_id_pattern):
    if not site_id_pattern: return "Unknown"
    match = site_id_pattern.search(trace_path)
    return match.group(1).upper() if match else "Unknown"

def extract_sector_number(trace_path):
    pattern = r'(?i)\b(s|sec|sect|sectr|sector)\s*[-_:#]*\s*([1-9]\d*)'
    match = re.search(pattern, trace_path)
    return int(match.group(2)) if match else 1

def extract_rf_band(trace_path):
    pattern = r'\b(1800|2100|2600|900|700|850|2300)\b'
    match = re.search(pattern, trace_path)
    return match.group(1) if match else "Unknown"

def extract_photo_status(trace_path):
    pre_pattern = r'(?i)\b(pre|befor|before|befr|initial|baseline|start|existing)\b'
    post_pattern = r'(?i)\b(post|after|aftr|final|done|mitigated|complete|complet)\b'
    if re.search(post_pattern, trace_path): return "Post"
    if re.search(pre_pattern, trace_path): return "Pre"
    return "Pre"

def extract_hardware_type(trace_path):
    pattern = r'(?i)\b(aau|rru)\b'
    match = re.search(pattern, trace_path)
    return match.group(1).upper() if match else "Unknown"

def analyze_photo_content_ai(trace_path, raw_bytes=None):
    photo_type = "Unidentified Image"
    path_lower = trace_path.lower()

    # Text-matching fallback overrides
    if re.search(r'(compass|azimuth|bearing|pano)', path_lower): photo_type = "Compass Dial"
    elif re.search(r'(inclinometer|tilt|level|bubble)', path_lower): photo_type = "Inclinometer Tilt"
    elif re.search(r'(tape|measure|measurement|height)', path_lower): photo_type = "Measurement Tape"

    if ML_AVAILABLE and raw_bytes and photo_type == "Unidentified Image":
        try:
            img = PILImage.open(io.BytesIO(raw_bytes))
            if img.mode != 'RGB': img = img.convert('RGB')
            img.thumbnail((800, 800), PILImage.Resampling.LANCZOS)
            
            prompt = (
                "Classify the main measurement tool shown in this image. "
                "Respond with EXACTLY ONE of these terms:\n"
                "- 'Compass Dial'\n- 'Inclinometer Tilt'\n- 'Measurement Tape'\n- 'Tower Equipment'\n"
            )
            # Ensure we never exceed 15 RPM free tier limit
            import time
            time.sleep(4.2)
            
            # Auto-retry logic for 429 errors
            from google.api_core.exceptions import ResourceExhausted
            for attempt in range(3):
                try:
                    response = model.generate_content([prompt, img])
                    break
                except ResourceExhausted:
                    print("⏳ Rate limit hit. Pausing for 30 seconds...")
                    time.sleep(30)

            output_text = response.text.strip().title()

            if 'Compass' in output_text: photo_type = "Compass Dial"
            elif 'Inclinometer' in output_text or 'Tilt' in output_text: photo_type = "Inclinometer Tilt"
            elif 'Tape' in output_text or 'Measure' in output_text: photo_type = "Measurement Tape"
            else: photo_type = "Tower Equipment"
            
        except Exception as e:
            print(f"⚠️ AI processing failed for {os.path.basename(trace_path)}: {e}")

    return photo_type


In [ ]:
# ==========================================
# 5. DIRECTORY SCANNING ENGINE
# ==========================================
def scan_inside_zip(zip_source, virtual_path, data_list, extract_dir, thumb_dir, site_id_pattern):
    try:
        with zipfile.ZipFile(zip_source, 'r') as z:
            for member in z.namelist():
                if member.endswith('/') or '__MACOSX' in member or os.path.basename(member).startswith('._'):
                    continue

                current_virtual_path = f"{virtual_path} -> {member}"
                ext = os.path.splitext(member)[1].lower()

                if ext in ['.jpg', '.jpeg', '.png']:
                    with z.open(member) as f:
                        raw_bytes = f.read()

                        override = get_container_overrides(current_virtual_path)
                        site_id = extract_site_id(current_virtual_path, site_id_pattern)
                        sector = extract_sector_number(current_virtual_path)
                        rf_band = override['band'] if override['band'] else extract_rf_band(current_virtual_path)
                        status = extract_photo_status(current_virtual_path)
                        hardware = override['hardware'] if override['hardware'] else extract_hardware_type(current_virtual_path)

                        photo_type = analyze_photo_content_ai(current_virtual_path, raw_bytes)
                        capture_date = extract_camera_capture_date(raw_bytes, zip_source)

                        clean_member_name = re.sub(r'[^a-zA-Z0-9_.-]', '_', os.path.basename(member))
                        export_filename = f"{site_id}_B{rf_band}_S{sector}_{status}_{clean_member_name}"
                        if not export_filename.lower().endswith('.jpg'): export_filename += '.jpg'
                        export_filepath = os.path.join(extract_dir, export_filename)

                        save_optimized_photo(raw_bytes, export_filepath)

                        thumb_filename = f"thumb_{site_id}_B{rf_band}_S{sector}_{status}_{clean_member_name}.webp"
                        thumb_filepath = os.path.join(thumb_dir, thumb_filename)
                        create_compressed_thumbnail(raw_bytes, thumb_filepath)

                        data_list.append({
                            'Site ID': site_id, 'Band': rf_band, 'Sector': sector,
                            'Status': status, 'Hardware': hardware, 'Type': photo_type,
                            'Capture Date': capture_date,
                            'photo_link_path': export_filepath, 'thumb_path': thumb_filepath
                        })
                elif ext == '.zip':
                    with z.open(member) as f:
                        nested_zip_bytes = io.BytesIO(f.read())
                        scan_inside_zip(nested_zip_bytes, current_virtual_path, data_list, extract_dir, thumb_dir, site_id_pattern)
    except Exception as e:
        print(f"❌ Critical error processing ZIP {zip_source}: {e}")

def process_photos_directory(target_root, extract_dir, thumb_dir, site_id_pattern):
    data_list = []
    for root, dirs, files in os.walk(target_root):
        if not os.path.abspath(root).startswith(os.path.abspath(target_root)):
            continue

        if any(bad in root for bad in ['Extracted_Photos', 'temp_thumbnails', '__MACOSX', 'Result']):
            continue

        for file in files:
            if file.startswith('._') or file.startswith('thumb_'):
                continue

            full_path = os.path.join(root, file)
            ext = os.path.splitext(file)[1].lower()

            if ext in ['.jpg', '.jpeg', '.png']:
                try:
                    override = get_container_overrides(full_path)
                    site_id = extract_site_id(full_path, site_id_pattern)
                    sector = extract_sector_number(full_path)
                    rf_band = override['band'] if override['band'] else extract_rf_band(full_path)
                    status = extract_photo_status(full_path)
                    hardware = override['hardware'] if override['hardware'] else extract_hardware_type(full_path)

                    with open(full_path, 'rb') as f:
                        raw_bytes = f.read()

                    photo_type = analyze_photo_content_ai(full_path, raw_bytes)
                    capture_date = extract_camera_capture_date(raw_bytes, full_path)

                    clean_member_name = re.sub(r'[^a-zA-Z0-9_.-]', '_', file)
                    export_filename = f"{site_id}_B{rf_band}_S{sector}_{status}_{clean_member_name}"
                    if not export_filename.lower().endswith('.jpg'): export_filename += '.jpg'
                    export_filepath = os.path.join(extract_dir, export_filename)

                    save_optimized_photo(raw_bytes, export_filepath)

                    thumb_filename = f"thumb_{site_id}_B{rf_band}_S{sector}_{status}_{clean_member_name}.webp"
                    thumb_filepath = os.path.join(thumb_dir, thumb_filename)
                    create_compressed_thumbnail(raw_bytes, thumb_filepath)

                    data_list.append({
                        'Site ID': site_id, 'Band': rf_band, 'Sector': sector,
                        'Status': status, 'Hardware': hardware, 'Type': photo_type,
                        'Capture Date': capture_date,
                        'photo_link_path': export_filepath, 'thumb_path': thumb_filepath
                    })
                    del raw_bytes
                    gc.collect()

                except Exception as e:
                    print(f"⚠️ Skipping file {file} due to error: {e}")
                    continue

            elif ext == '.zip':
                scan_inside_zip(full_path, full_path, data_list, extract_dir, thumb_dir, site_id_pattern)
                gc.collect()
    return data_list


In [ ]:
# ==========================================
# 6. EXECUTE PIPELINE
# ==========================================
if not os.path.exists(PREFIX_FILE):
    print(f"❌ ERROR: Prefix.txt not found at {PREFIX_FILE}.")
else:
    with open(PREFIX_FILE, 'r') as pf:
        raw_lines = pf.read().splitlines()

    site_prefixes = [p.strip() for p in raw_lines if p.strip() and p.strip().lower() != 'prefix']
    if "17-Apr" in site_prefixes and "17APR" not in site_prefixes:
        site_prefixes.append("17APR")
        
    escaped_prefixes = [re.escape(p) for p in site_prefixes]
    prefix_group = '|'.join(escaped_prefixes)
    site_id_pattern = re.compile(rf'(({prefix_group})[a-zA-Z0-9]{{4}})', re.IGNORECASE)

    print(f"🔍 Scanning '{TARGET_WORKSPACE}'...")
    extracted_data = process_photos_directory(TARGET_WORKSPACE, EXTRACT_PHOTOS_DIR, THUMB_CACHE_DIR, site_id_pattern)
    df = pd.DataFrame(extracted_data)
    
    if df.empty:
        print("❌ ERROR: No usable audit images matched your pattern in the target folder.")
    else:
        print(f"✅ Successfully identified {len(df)} images. Assembling Matrix...")
        df['key'] = df['Site ID'].astype(str) + "_" + df['Sector'].astype(str) + "_" + df['Band'].astype(str) + "_" + df['Type'].astype(str)

        pre = df[df['Status'] == 'Pre'].set_index('key')
        post = df[df['Status'] == 'Post'].set_index('key')
        merged = pre.join(post, lsuffix='_pre', rsuffix='_post', how='outer').reset_index()

        merged_data = []
        for _, row in merged.iterrows():
            merged_data.append({
                'Site ID': row.get('Site ID_pre') if pd.notna(row.get('Site ID_pre')) else row.get('Site ID_post'),
                'Sector': row.get('Sector_pre') if pd.notna(row.get('Sector_pre')) else row.get('Sector_post'),
                'Band': row.get('Band_pre') if pd.notna(row.get('Band_pre')) else row.get('Band_post'),
                'Hardware_pre': row.get('Hardware_pre', ''),
                'thumb_pre': row.get('thumb_path_pre', ''),
                'Type_pre': row.get('Type_pre', ''),
                'link_pre': row.get('photo_link_path_pre', ''),
                'Hardware_post': row.get('Hardware_post', ''),
                'thumb_post': row.get('thumb_path_post', ''),
                'Type_post': row.get('Type_post', ''),
                'link_post': row.get('photo_link_path_post', '')
            })

        df_merged = pd.DataFrame(merged_data)
        df_merged.fillna('', inplace=True)
        df_merged.sort_values(by=['Site ID', 'Sector', 'Type_pre'], inplace=True)

        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "AOR_Report"

        header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
        dark_blue_fill = PatternFill(start_color='2C3E50', end_color='2C3E50', fill_type='solid')
        light_blue_fill = PatternFill(start_color='0070C0', end_color='0070C0', fill_type='solid')
        data_font = Font(name='Segoe UI', size=10)
        link_font = Font(name='Segoe UI', size=10, color='1A5276', underline='single')
        center_align = Alignment(horizontal='center', vertical='center')
        all_borders = Border(left=Side(style='thin', color='BDC3C7'), right=Side(style='thin', color='BDC3C7'), top=Side(style='thin', color='BDC3C7'), bottom=Side(style='thin', color='BDC3C7'))

        ws['A1'] = "Site ID"; ws.merge_cells('A1:A2')
        ws['B1'] = "Sector"; ws.merge_cells('B1:B2')
        ws['C1'] = "Band"; ws.merge_cells('C1:C2')
        ws['D1'] = "Pre"; ws.merge_cells('D1:G1')
        ws['H1'] = "Post"; ws.merge_cells('H1:K1')

        sub_headers = ["Hardware", "Photo Preview", "Type", "Full Resolution Link",
                       "Hardware", "Photo Preview", "Type", "Full Resolution Link"]
        for col_idx, header in enumerate(sub_headers, start=4):
            ws.cell(row=2, column=col_idx, value=header)

        for row_idx, row_data in enumerate(df_merged.to_dict(orient='records'), start=3):
            ws.cell(row=row_idx, column=1, value=row_data.get('Site ID', ''))
            ws.cell(row=row_idx, column=2, value=row_data.get('Sector', ''))
            ws.cell(row=row_idx, column=3, value=row_data.get('Band', ''))

            # PRE
            ws.cell(row=row_idx, column=4, value=row_data.get('Hardware_pre', ''))
            thumb_pre = row_data.get('thumb_pre')
            if isinstance(thumb_pre, str) and os.path.exists(thumb_pre):
                xl_img_pre = openpyxl.drawing.image.Image(thumb_pre)
                xl_img_pre.width = 100; xl_img_pre.height = 75
                ws.add_image(xl_img_pre, f"E{row_idx}")
            ws.cell(row=row_idx, column=6, value=row_data.get('Type_pre', ''))
            link_pre = row_data.get('link_pre')
            if isinstance(link_pre, str) and link_pre:
                rel_link_pre = f'Extracted_Photos/{os.path.basename(link_pre)}'
                ws.cell(row=row_idx, column=7, value=f'=HYPERLINK("{rel_link_pre}", "Open Link 📸")')

            # POST
            ws.cell(row=row_idx, column=8, value=row_data.get('Hardware_post', ''))
            thumb_post = row_data.get('thumb_post')
            if isinstance(thumb_post, str) and os.path.exists(thumb_post):
                xl_img_post = openpyxl.drawing.image.Image(thumb_post)
                xl_img_post.width = 100; xl_img_post.height = 75
                ws.add_image(xl_img_post, f"I{row_idx}")
            ws.cell(row=row_idx, column=10, value=row_data.get('Type_post', ''))
            link_post = row_data.get('link_post')
            if isinstance(link_post, str) and link_post:
                rel_link_post = f'Extracted_Photos/{os.path.basename(link_post)}'
                ws.cell(row=row_idx, column=11, value=f'=HYPERLINK("{rel_link_post}", "Open Link 📸")')

        col_widths = {'A':15, 'B':8, 'C':10, 'D':12, 'E':16, 'F':20, 'G':20, 'H':12, 'I':16, 'J':20, 'K':20}
        for col, width in col_widths.items(): ws.column_dimensions[col].width = width

        for row_num in range(1, ws.max_row + 1):
            ws.row_dimensions[row_num].height = 62 if row_num > 2 else 25
            for col_num in range(1, 12):
                cell = ws.cell(row=row_num, column=col_num)
                cell.border, cell.alignment = all_borders, center_align
                if row_num in [1, 2]:
                    cell.font = header_font
                    cell.fill = dark_blue_fill if col_num <= 7 else light_blue_fill
                else:
                    cell.font = data_font if col_num not in [7, 11] else link_font

        report_filename = os.path.join(RESULT_DIR, f"AOR_Report_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.xlsx")
        wb.save(report_filename)
        wb.close()

        print("\n📦 Compiling clean asset package zip archive...")
        zip_filename = os.path.join(RESULT_DIR, f"AOR_Deliverable_{datetime.now().strftime('%Y-%m-%d_%H-%M')}.zip")

        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(report_filename, arcname=os.path.basename(report_filename))
            for root_dir, _, files in os.walk(EXTRACT_PHOTOS_DIR):
                for file in files:
                    full_file_path = os.path.join(root_dir, file)
                    arc_name = os.path.join("Extracted_Photos", file)
                    zipf.write(full_file_path, arcname=arc_name)

        print(f"\n🎉 DATA MATCH COMPLETE! Report compiled in your Result directory:")
        print(f"👉 {zip_filename}")
        
        print("\n⬇️ Automatically downloading the Deliverable to your computer...")
        files.download(zip_filename)
